## Processing clean data

This notebook processes clean data into statistics:

- Rolling statistics: form features from recent matches
- Elo rating: estimate team strength based on performance

In [2]:
import pandas as pd
from pathlib import Path

In [3]:
project_route = Path.cwd().parent
csv_route = project_route / "data" / "raw" / "E0.csv"
df = pd.read_csv(csv_route, encoding="latin1")
df.columns = df.columns.astype(str).str.replace("ï»¿", "", regex=False).str.strip()

In [ ]:
all_columns = {
    'div': 'league_division',
    'date': 'date',
    'time': 'kickoff_time',
    'hometeam': 'home_team',
    'awayteam': 'away_team',
    'referee': 'referee',
    'attendance': 'attendance',
    'ftr': 'result',
    'fthg': 'home_goals',
    'ftag': 'away_goals',
    'hthg': 'half_time_home_goals',
    'htag': 'half_time_away_goals',
    'htr': 'half_time_result',
    'hs': 'home_shots',
    'as': 'away_shots',
    'hst': 'home_shots_on_target',
    'ast': 'away_shots_on_target',
    'hhw': 'home_hit_woodwork',
    'ahw': 'away_hit_woodwork',
    'hc': 'home_corners',
    'ac': 'away_corners',
    'hf': 'home_fouls_committed',
    'af': 'away_fouls_committed',
    'hfkc': 'home_free_kicks_conceded',
    'afkc': 'away_free_kicks_conceded',
    'ho': 'home_offsides',
    'ao': 'away_offsides',
    'hy': 'home_yellow',
    'ay': 'away_yellow',
    'hr': 'home_red',
    'ar': 'away_red',
    'b365h': 'b365_home_win_odds',
    'b365d': 'b365_draw_odds',
    'b365a': 'b365_away_win_odds',
    'avgh': 'market_avg_home_win_odds',
    'avgd': 'market_avg_draw_odds',
    'avga': 'market_avg_away_win_odds',
}

lower_case_columns = {column.lower(): column for column in df.columns}
selected_columns = [lower_case_columns[column] for column in all_columns if column in lower_case_columns]
df = df[selected_columns]
df.rename(columns = {
    lower_case_columns[column]: all_columns[column]
    for column in all_columns if column in lower_case_columns
}, inplace=True)

df['date'] = pd.to_datetime(df['date'], dayfirst=True, errors='coerce')
df['kickoff_time'] = df['kickoff_time'].astype('string').str.strip()
df['match_datetime'] = pd.to_datetime(df['date'].dt.strftime('%Y-%m-%d') + ' ' + df['kickoff_time'], errors = 'coerce')
df.drop(columns = ['date', 'kickoff_time'], errors = 'ignore', inplace=True)

text_columns = [
    'league_division',
    'home_team',
    'away_team',
    'referee',
    'result',
    'half_time_result'
]
for column in text_columns:
    if column in df.columns:
        df[column] = df[column].astype('string').str.strip()

numeric_columns = [
    'attendance',
    'home_goals',
    'away_goals',
    'half_time_home_goals',
    'half_time_away_goals',
    'b365_home_win_odds',
    'b365_draw_odds',
    'b365_away_win_odds',
    'market_avg_home_win_odds',
    'market_avg_draw_odds',
    'market_avg_away_win_odds',
    'home_shots',
    'away_shots',
    'home_shots_on_target',
    'away_shots_on_target',
    'home_hit_woodwork',
    'away_hit_woodwork',
    'home_corners',
    'away_corners',
    'home_fouls_committed',
    'away_fouls_committed',
    'home_free_kicks_conceded',
    'away_free_kicks_conceded',
    'home_offsides',
    'away_offsides',
    'home_yellow',
    'away_yellow',
    'home_red',
    'away_red',
]
for column in numeric_columns:
    if column in df.columns:
        df[column] = pd.to_numeric(df[column], errors='coerce')

df = (df.dropna(subset=['match_datetime', 'home_team', 'away_team']).sort_values(
    ['match_datetime', 'home_team', 'away_team']
    ).reset_index(drop=True)
)

In [ ]:
clean_frame = df.copy()
window = 5

df_home = clean_frame[
    [
        "match_datetime",
        "home_team",
        "away_team",
        "result",
        "home_goals",
        "away_goals",
        "home_shots",
        "home_shots_on_target",
        "home_corners",
        "home_fouls_committed",
        "home_yellow",
        "home_red",
    ]
].copy().rename(columns={
    "home_team": "team",
    "away_team": "opponent",
    "result": "points",
    "home_goals": "goals_scored",
    "away_goals": "goals_conceded",
    "home_shots": "shots",
    "home_shots_on_target": "shots_on_target",
    "home_corners": "corners",
    "home_fouls_committed": "fouls_committed",
    "home_yellow": "yellow",
    "home_red": "red",
})
df_home["points"] = df_home["points"].map({"H": 3, "D": 1, "A": 0})

df_away = clean_frame[
    [
        "match_datetime",
        "home_team",
        "away_team",
        "result",
        "home_goals",
        "away_goals",
        "away_shots",
        "away_shots_on_target",
        "away_corners",
        "away_fouls_committed",
        "away_yellow",
        "away_red",
    ]
].copy().rename(columns={
    "away_team": "team",
    "home_team": "opponent",
    "result": "points",
    "away_goals": "goals_scored",
    "home_goals": "goals_conceded",
    "away_shots": "shots",
    "away_shots_on_target": "shots_on_target",
    "away_corners": "corners",
    "away_fouls_committed": "fouls_committed",
    "away_yellow": "yellow",
    "away_red": "red",
})
df_away["points"] = df_away["points"].map({"A": 3, "D": 1, "H": 0})

df_home["is_home"] = True
df_away["is_home"] = False
long_df = pd.concat([df_home, df_away], ignore_index=True)
long_df = long_df.sort_values(by=["team", "match_datetime"]).reset_index(drop=True)
long_df.head()

In [ ]:
window = 5

features = [
    "goals_scored",
    "goals_conceded",
    "shots",
    "shots_on_target",
    "corners",
    "points"
]

def add_rolling_average(clean_frame, features, window):

    df_home = clean_frame[
        [
            "match_datetime",
            "home_team",
            "away_team",
            "result",
            "home_goals",
            "away_goals",
            "home_shots",
            "home_shots_on_target",
            "home_corners",
            "home_fouls_committed",
            "home_yellow",
            "home_red",
        ]
    ].copy().rename(columns={
        "home_team": "team",
        "away_team": "opponent",
        "result": "points",
        "home_goals": "goals_scored",
        "away_goals": "goals_conceded",
        "home_shots": "shots",
        "home_shots_on_target": "shots_on_target",
        "home_corners": "corners",
        "home_fouls_committed": "fouls_committed",
        "home_yellow": "yellow",
        "home_red": "red",
    })

    df_home["points"] = df_home["points"].map({"H": 3, "D": 1, "A": 0})

    df_away = clean_frame[
        [
            "match_datetime",
            "home_team",
            "away_team",
            "result",
            "home_goals",
            "away_goals",
            "away_shots",
            "away_shots_on_target",
            "away_corners",
            "away_fouls_committed",
            "away_yellow",
            "away_red",
        ]
    ].copy().rename(columns={
        "away_team": "team",
        "home_team": "opponent",
        "result": "points",
        "away_goals": "goals_scored",
        "home_goals": "goals_conceded",
        "away_shots": "shots",
        "away_shots_on_target": "shots_on_target",
        "away_corners": "corners",
        "away_fouls_committed": "fouls_committed",
        "away_yellow": "yellow",
        "away_red": "red",
    })

    df_away["points"] = df_away["points"].map({"A": 3, "D": 1, "H": 0})

    df_home["is_home"] = True
    df_away["is_home"] = False

    long_df = pd.concat([df_home, df_away], ignore_index=True)

    long_df = (
        long_df
        .sort_values(["team", "match_datetime"])
        .reset_index(drop=True)
    )

    for feature in features:

        previous = long_df.groupby("team")[feature].shift(1)

        long_df[f"avg_{feature}_last_{window}_matches"] = (
            previous
            .groupby(long_df["team"])
            .rolling(window=window, min_periods=1)
            .mean()
            .reset_index(level=0, drop=True)
            .round(2)
        )

    rolling_columns = [
        f"avg_{feature}_last_{window}_matches"
        for feature in features
    ]

    home_features = (
        long_df[long_df["is_home"]]
        [["match_datetime", "team"] + rolling_columns]
        .rename(
            columns={
                "team": "home_team",
                **{
                    col: f"{col}_home"
                    for col in rolling_columns
                }
            }
        )
    )

    away_features = (
        long_df[~long_df["is_home"]]
        [["match_datetime", "team"] + rolling_columns]
        .rename(
            columns={
                "team": "away_team",
                **{
                    col: f"{col}_away"
                    for col in rolling_columns
                }
            }
        )
    )

    clean_frame = clean_frame.merge(
        home_features,
        on=["match_datetime", "home_team"],
        how="left"
    )

    clean_frame = clean_frame.merge(
        away_features,
        on=["match_datetime", "away_team"],
        how="left"
    )

    return clean_frame

['league_division', 'home_team', 'away_team', 'referee', 'result', 'home_goals', 'away_goals', 'half_time_home_goals', 'half_time_away_goals', 'half_time_result', 'home_shots', 'away_shots', 'home_shots_on_target', 'away_shots_on_target', 'home_corners', 'away_corners', 'home_fouls_committed', 'away_fouls_committed', 'home_yellow', 'away_yellow', 'home_red', 'away_red', 'b365_home_win_odds', 'b365_draw_odds', 'b365_away_win_odds', 'market_avg_home_win_odds', 'market_avg_draw_odds', 'market_avg_away_win_odds', 'match_datetime']
